# 06 V2 — Synthetic Data Generation (Gemini Teacher)

Generates high-quality schema.org JSON-LD training examples using **Gemini 2.5 Flash** as a teacher model.

## Pipeline
1. **Input**: Irish SME webpages (HTML + screenshot) from Common Crawl / WDC
2. **Generate**: Gemini 2.5 Flash produces JSON-LD for each page
3. **Validate**: Three-stage validation (structure → vocab → factual)
4. **Rescore** *(optional)*: Gemini 2.5 Pro judges quality 1-5
5. **Export**: JSONL training data for fine-tuning

## Cost estimate
| Mode | Cost per example | 5K examples |
|------|-----------------|-------------|
| Standard (Flash) | ~$0.002 | ~$10 |
| Batch (Flash, -50%) | ~$0.001 | ~$5 |
| + Rescore 10% (Pro) | +$0.003/example | +$1.50 |

**Total target: ~$50–90 for 5K–7K validated examples**

In [ ]:
import sys
import os
from pathlib import Path

# Setup paths
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=True)

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
assert GEMINI_API_KEY, 'GEMINI_API_KEY not set in .env'

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'v2_synthetic'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results dir:  {RESULTS_DIR}')

## 1. Prepare Input Pages

Load Irish SME pages (HTML + screenshot paths) as input for the generation pipeline.
Pages come from the Common Crawl extraction in notebooks 01 / 03.

In [ ]:
import json
from collections import Counter

# Load pages — expects JSONL with fields: url, html, screenshot_path
PAGES_PATH = PROCESSED_DIR / 'pages_for_generation.jsonl'

pages = []
if PAGES_PATH.exists():
    with open(PAGES_PATH) as f:
        for line in f:
            pages.append(json.loads(line))
    print(f'Loaded {len(pages):,} pages from {PAGES_PATH}')
else:
    print(f'Pages file not found: {PAGES_PATH}')
    print('Run notebooks 01/03 first, or point PAGES_PATH at your input JSONL.')

# Quick domain breakdown
if pages:
    from urllib.parse import urlparse
    tlds = Counter(urlparse(p['url']).netloc.split('.')[-1] for p in pages)
    print('TLD breakdown:', dict(tlds.most_common(5)))

## 2. Configure Generation

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from generate import PipelineConfig

config = PipelineConfig(
    gemini_api_key=GEMINI_API_KEY,
    model="gemini-2.5-flash-preview-04-17",
    mode="batch",            # 'batch' (cheaper, slower) or 'standard'
    concurrency=10,          # parallel requests (standard mode only)
    max_retries=3,
    system_prompt_path=str(PROJECT_ROOT / 'prompts' / 'teacher_system_prompt.txt'),
    type_distribution_path=str(PROJECT_ROOT / 'config' / 'type_distribution.json'),
    output_dir=str(RESULTS_DIR),
    max_html_chars=12000,    # trim HTML to fit token budget
)

print(f'Mode: {config.mode}')
print(f'Model: {config.model}')
print(f'Output: {config.output_dir}')

## 3. Run Generation

Generates JSON-LD for each page. Results saved to `results/v2_synthetic/results.jsonl`.
Safe to interrupt and resume — already-completed pages are skipped.

In [ ]:
from generate import Pipeline

# For a quick test run, limit to first N pages
TEST_RUN = True
N_TEST   = 20

input_pages = pages[:N_TEST] if TEST_RUN else pages
print(f'Generating for {len(input_pages)} pages (test={TEST_RUN})')

pipeline = Pipeline(config)
results  = pipeline.run(input_pages)

valid    = [r for r in results if r.valid]
invalid  = [r for r in results if not r.valid]
print(f'\nResults: {len(valid)} valid / {len(invalid)} invalid / {len(results)} total')
print(f'Pass rate: {len(valid)/len(results)*100:.1f}%' if results else 'No results')

## 4. Validate Results

Run the three-stage schema validator over all generated outputs.

In [ ]:
from validators.schema_validator import validate
from collections import Counter

# Re-validate all results from disk (in case resuming)
RESULTS_JSONL = RESULTS_DIR / 'results.jsonl'

all_results = []
if RESULTS_JSONL.exists():
    with open(RESULTS_JSONL) as f:
        for line in f:
            all_results.append(json.loads(line))

print(f'Total results on disk: {len(all_results)}')

# Validation breakdown
stage_failures = Counter()
type_counts    = Counter()
valid_count    = 0

for r in all_results:
    vr = validate(r.get('generated_jsonld', ''), r.get('html', ''))
    if vr.valid:
        valid_count += 1
    for issue in vr.issues:
        if issue.severity == 'error':
            stage_failures[issue.category] += 1

print(f'Valid: {valid_count}/{len(all_results)} ({valid_count/len(all_results)*100:.1f}%)' if all_results else 'No results')
print('Top failure categories:', stage_failures.most_common(5))

## 5. Optional: Premium Rescore (Gemini 2.5 Pro)

Spot-check ~10% of valid outputs with Gemini 2.5 Pro as quality judge.
Scores 1-5 on: type_accuracy, property_completeness, factual_accuracy, structural_validity, schema_compliance.

In [ ]:
RUN_RESCORE = False  # Set True to run (costs ~$0.003/example)

if RUN_RESCORE:
    import subprocess, sys
    rescore_script = PROJECT_ROOT / 'scripts' / 'rescore.py'
    cmd = [
        sys.executable, str(rescore_script),
        '--results', str(RESULTS_JSONL),
        '--pages',   str(PAGES_PATH),
        '--output',  str(RESULTS_DIR / 'rescored.jsonl'),
        '--sample',  '0.1',       # rescore 10% of valid examples
        '--min-score', '3.5',
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    print('Rescore skipped (RUN_RESCORE=False)')

## 6. Export Training Data

Export validated examples to JSONL format compatible with the fine-tuning pipeline (notebook 07).

In [ ]:
from generate import Pipeline
import subprocess, sys

EXPORT_PATH = PROCESSED_DIR / 'train_v2.jsonl'

export_cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'generate.py'),
    'export',
    '--results', str(RESULTS_JSONL),
    '--output',  str(EXPORT_PATH),
    '--min-score', '0',   # adjust if using rescore
]

result = subprocess.run(export_cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

if EXPORT_PATH.exists():
    with open(EXPORT_PATH) as f:
        n = sum(1 for _ in f)
    size_mb = EXPORT_PATH.stat().st_size / 1e6
    print(f'\nExported {n:,} examples → {EXPORT_PATH} ({size_mb:.1f} MB)')

## 7. Summary Stats

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image as IPImage

# Load exported training data
train_examples = []
if EXPORT_PATH.exists():
    with open(EXPORT_PATH) as f:
        for line in f:
            train_examples.append(json.loads(line))

print(f'Training examples: {len(train_examples):,}')

if train_examples:
    # Extract @type from each example's output
    types = []
    for ex in train_examples:
        try:
            output = ex.get('output', '')
            if isinstance(output, str):
                parsed = json.loads(output)
            else:
                parsed = output
            if isinstance(parsed, list):
                parsed = parsed[0]
            t = parsed.get('@type', 'Unknown')
            types.append(t if isinstance(t, str) else t[0] if t else 'Unknown')
        except Exception:
            types.append('ParseError')

    type_counts = Counter(types)
    top_types   = type_counts.most_common(15)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('V2 Synthetic Training Data', fontsize=14, fontweight='bold')

    # Type distribution bar chart
    labels, counts = zip(*top_types)
    ax1.barh(range(len(labels)), counts, color='steelblue')
    ax1.set_yticks(range(len(labels)))
    ax1.set_yticklabels(labels, fontsize=9)
    ax1.set_xlabel('Count')
    ax1.set_title('@type Distribution (top 15)')
    ax1.invert_yaxis()

    # Load target distribution for comparison
    target_path = PROJECT_ROOT / 'config' / 'type_distribution.json'
    if target_path.exists():
        with open(target_path) as f:
            target = json.load(f)
        target_weights = {k: v['weight'] for k, v in target['types'].items()}
        actual_pct  = {t: c/len(types)*100 for t, c in type_counts.items()}
        common_types = sorted(set(target_weights) & set(actual_pct))
        if common_types:
            x = range(len(common_types))
            ax2.bar([i - 0.2 for i in x], [target_weights[t] for t in common_types],
                    width=0.4, label='Target %', color='lightcoral', alpha=0.8)
            ax2.bar([i + 0.2 for i in x], [actual_pct.get(t, 0) for t in common_types],
                    width=0.4, label='Actual %', color='steelblue', alpha=0.8)
            ax2.set_xticks(list(x))
            ax2.set_xticklabels(common_types, rotation=45, ha='right', fontsize=8)
            ax2.set_ylabel('%')
            ax2.set_title('Actual vs Target Distribution')
            ax2.legend()

    plt.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    fig.savefig(RESULTS_DIR / 'type_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    display(IPImage(data=buf.getvalue()))
    print(f'Chart saved to {RESULTS_DIR}/type_distribution.png')
else:
    print('No training examples to visualise yet.')